<a href="https://colab.research.google.com/github/uditmodi2322-bot/Gold-predictions/blob/main/gold_pred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from sklearn.ensemble import RandomForestRegressor
from datetime import datetime
import warnings

# Suppress warnings for a cleaner output
warnings.filterwarnings("ignore", category=FutureWarning)

# --- 1. CONFIGURATION & CONVERSIONS ---
USD_TO_INR = 92.61
OZ_TO_GRAM = 31.1035

# --- 2. FETCH LIVE DATA ---
print("Fetching live gold market data...")
# Explicitly set auto_adjust=True to fix the FutureWarning
gold_data = yf.download("GC=F", start="2015-01-01", end=datetime.now().strftime('%Y-%m-%d'), auto_adjust=True)

# --- 3. PREPARE DATA ---
df = gold_data[['Close']].copy()
# yfinance returns a MultiIndex sometimes; we flatten it to ensure 'Close' is a simple Series
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df.reset_index(inplace=True)

# Create Time Features
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

# --- 4. TRAIN MODEL ---
X = df[['Year', 'Month', 'Day']]
# Use .values.ravel() to convert the column vector to a 1D array to fix the DataConversionWarning
y = df['Close'].values.ravel()

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

# --- 5. PREDICTION & CONVERSION ---
target_date = "2026-04-14"
dt_obj = datetime.strptime(target_date, '%Y-%m-%d')

# Create a DataFrame for prediction to match feature names and avoid warnings
predict_df = pd.DataFrame([[dt_obj.year, dt_obj.month, dt_obj.day]], columns=['Year', 'Month', 'Day'])
pred_usd_oz = model.predict(predict_df)[0]

# Convert USD/Ounce to INR/Gram
pred_inr_gram = (pred_usd_oz / OZ_TO_GRAM) * USD_TO_INR

# --- 6. OUTPUT ---
print("\n" + "="*45)
print(f"GOLD PRICE FORECAST FOR: {target_date}")
print("-" * 45)
print(f"Price (USD / Ounce): ${pred_usd_oz:,.2f}")
print(f"Price (INR / Gram):  ₹{pred_inr_gram:,.2f}")
print(f"Price (INR / 10g):   ₹{pred_inr_gram * 10:,.2f}")
print("="*45)

Fetching live gold market data...


[*********************100%***********************]  1 of 1 completed



GOLD PRICE FORECAST FOR: 2026-04-14
---------------------------------------------
Price (USD / Ounce): $4,797.84
Price (INR / Gram):  ₹14,285.48
Price (INR / 10g):   ₹142,854.80


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from sklearn.tree import DecisionTreeRegressor  # Using Decision Tree specifically
from datetime import datetime
import warnings

# Suppress warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# --- 1. CONFIGURATION ---
USD_TO_INR = 92.61
OZ_TO_GRAM = 31.1035

# --- 2. FETCH DATA ---
print("Fetching live gold market data...")
gold_data = yf.download("GC=F", start="2015-01-01", end=datetime.now().strftime('%Y-%m-%d'), auto_adjust=True)

# --- 3. PREPARE DATA ---
df = gold_data[['Close']].copy()
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df.reset_index(inplace=True)
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

X = df[['Year', 'Month', 'Day']]
y = df['Close'].values.ravel()

# --- 4. TRAIN DECISION TREE MODEL ---
# Decision Trees split data into branches to reach a prediction.
model = DecisionTreeRegressor(random_state=42)
model.fit(X, y)

# --- 5. PREDICTION ---
target_date = "2026-04-14"
dt_obj = datetime.strptime(target_date, '%Y-%m-%d')
predict_df = pd.DataFrame([[dt_obj.year, dt_obj.month, dt_obj.day]], columns=['Year', 'Month', 'Day'])

pred_usd_oz = model.predict(predict_df)[0]

# Convert USD/Ounce to INR/10 Grams
pred_inr_gram = (pred_usd_oz / OZ_TO_GRAM) * USD_TO_INR
pred_inr_10g = pred_inr_gram * 10

# --- 6. OUTPUT ---
print("\n" + "="*45)
print(f"DECISION TREE FORECAST FOR: {target_date}")
print("-" * 45)
print(f"Price (USD / Ounce): ${pred_usd_oz:,.2f}")
print(f"Price (INR / 10g):   ₹{pred_inr_10g:,.2f}")
print("="*45)

[*********************100%***********************]  1 of 1 completed

Fetching live gold market data...

DECISION TREE FORECAST FOR: 2026-04-14
---------------------------------------------
Price (USD / Ounce): $4,825.00
Price (INR / 10g):   ₹143,663.33


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from datetime import datetime, timedelta

# --- 1. SETTINGS & CONVERSIONS ---
# Using your specific rate of 94.89
USD_TO_INR = 92.61
# 1 Troy Ounce = 31.1035 Grams (International Standard for 24K Gold)
OZ_TO_GRAM = 31.1035

# Set timeframe for clustering (last 2 years to get enough data points)
target_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=730)).strftime('%Y-%m-%d')

print(f"Fetching 24K Gold Market Data (USD/INR @ {USD_TO_INR})...")

# --- 2. DATA ACQUISITION & CLEANING ---
# auto_adjust=True helps prevent header errors
raw_data = yf.download("GC=F", start=start_date, end=target_date, auto_adjust=True)

# FIX: Flatten MultiIndex if yfinance returned nested columns
if isinstance(raw_data.columns, pd.MultiIndex):
    raw_data.columns = raw_data.columns.get_level_values(0)

# Create a clean DataFrame
df = pd.DataFrame(index=raw_data.index)
df['Close_USD'] = raw_data['Close'].astype(float)

# Conversion Logic: (Price / 31.1035) * 94.89
df['INR_per_Gram'] = (df['Close_USD'] / OZ_TO_GRAM) * USD_TO_INR
df['Volatility'] = df['Close_USD'].pct_change()
df = df.dropna()

# --- 3. CLUSTERING ENGINE ---
# Features for K-Means: Price and Daily Movement
X = df[['INR_per_Gram', 'Volatility']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3 Clusters: Low Value, Mid Range, and Peak (Regimes)
model = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster'] = model.fit_predict(X_scaled)

# --- 4. SCALAR EXTRACTION (Fixes the TypeError) ---
latest_row = df.iloc[-1]
current_price = float(latest_row['INR_per_Gram'])
current_cluster = int(latest_row['Cluster'])

# Identify High/Low clusters automatically
means = df.groupby('Cluster')['INR_per_Gram'].mean().sort_values()
low_id, high_id = means.index[0], means.index[-1]

# --- 5. FINAL REPORT ---
print("\n" + "="*45)
print(f"24K GOLD ANALYSIS: {datetime.now().strftime('%d-%m-%Y')}")
print("="*45)
print(f"Price per Gram (24K):  ₹{current_price:,.2f}")
print(f"Price per 10g (24K):   ₹{current_price * 10:,.2f}")
print(f"Market Regime ID:      Cluster {current_cluster}")

# Logic for user status
if current_cluster == high_id:
    status = "⚠️ BULLISH PEAK (Historically High Zone)"
elif current_cluster == low_id:
    status = "✅ ACCUMULATION (Historically Low Zone)"
else:
    status = "⚖️ NEUTRAL (Mid-Market Range)"

print(f"Market Status:         {status}")
print("="*45)

Fetching 24K Gold Market Data (USD/INR @ 92.61)...


[*********************100%***********************]  1 of 1 completed


24K GOLD ANALYSIS: 16-06-2026
Price per Gram (24K):  ₹12,886.53
Price per 10g (24K):   ₹128,865.27
Market Regime ID:      Cluster 0
Market Status:         ⚠️ BULLISH PEAK (Historically High Zone)
